# Unificación y limpieza inicial de los datasets

Se realiza la construcción del dataset unificado a partir de los seis subconjuntos seleccionados del dataset LITNET-2020. Debido al tamaño de los archivos CSV, la lectura se realiza por bloques/chunks para evitar cargar todos los registros simultáneamente en memoria.

Durante este proceso se asignan nombres temporales a las variables, se identifican las etiquetas `attack_t` y `attack_a`, se agrega información sobre el archivo de origen y la familia general del ataque, y se guarda una versión unificada en formato Parquet para ser utilizada en las siguientes etapas del proyecto.

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

In [2]:
RAW_PATH = Path("../data/raw")
PROCESSED_PATH = Path("../data/processed")
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
files = {
    "BLASTER_WORM_v2.csv": "worm",
    "REAPER_WORM_v2.csv": "worm",
    "RED_WORM_v2.csv": "worm",
    "HTTP_FLOOD_v2.csv": "flood",
    "ICMP_FLOOD_v2.csv": "flood",
    "UDP_FLOOD_v2.csv": "flood"
}
column_names = [f"feature_{i+1}" for i in range(83)] + ["attack_t", "attack_a"]

### prueba

In [3]:
def leer_muestra_archivo(file_name, attack_family, nrows=10000):
    path = RAW_PATH / file_name
    df = pd.read_csv(
        path,
        header=None,
        names=column_names,
        nrows=nrows
    )
    df["dataset_name"] = file_name.replace(".csv", "")
    df["attack_family"] = attack_family
    return df

In [4]:
muestras = []
for file_name, attack_family in files.items():
    print(f"Leyendo muestra de: {file_name}")
    df_temp = leer_muestra_archivo(file_name, attack_family, nrows=10000)
    muestras.append(df_temp)
df_muestra_unificada = pd.concat(muestras, ignore_index=True)
df_muestra_unificada.shape

Leyendo muestra de: BLASTER_WORM_v2.csv
Leyendo muestra de: REAPER_WORM_v2.csv
Leyendo muestra de: RED_WORM_v2.csv
Leyendo muestra de: HTTP_FLOOD_v2.csv
Leyendo muestra de: ICMP_FLOOD_v2.csv
Leyendo muestra de: UDP_FLOOD_v2.csv


(60000, 87)

In [7]:
df_muestra_unificada.sample(17)

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_78,feature_79,feature_80,feature_81,feature_82,feature_83,attack_t,attack_a,dataset_name,attack_family
24871,4872,2020,1,30,20,31,7,2020,1,30,...,normal,normal,normal,normal,normal,normal,none,0,RED_WORM_v2,worm
39913,9914,2020,1,16,9,28,57,2020,1,16,...,normal,normal,normal,normal,normal,normal,none,0,HTTP_FLOOD_v2,flood
5306,5307,2020,1,25,3,43,57,2020,1,25,...,normal,normal,normal,normal,normal,normal,none,0,BLASTER_WORM_v2,worm
37732,7733,2020,1,16,9,28,57,2020,1,16,...,normal,normal,normal,normal,normal,normal,none,0,HTTP_FLOOD_v2,flood
47849,7850,2020,1,31,15,34,59,2020,1,31,...,normal,normal,normal,normal,normal,normal,none,0,ICMP_FLOOD_v2,flood
50992,993,2019,3,6,21,13,58,2019,3,6,...,normal,normal,normal,normal,normal,normal,none,0,UDP_FLOOD_v2,flood
34055,4056,2020,1,16,9,28,57,2020,1,16,...,normal,normal,normal,normal,normal,normal,none,0,HTTP_FLOOD_v2,flood
7029,7030,2020,1,25,3,43,57,2020,1,25,...,normal,normal,normal,normal,normal,normal,none,0,BLASTER_WORM_v2,worm
58444,8445,2019,3,6,21,14,3,2019,3,6,...,normal,normal,normal,normal,normal,normal,none,0,UDP_FLOOD_v2,flood
31142,1143,2020,1,16,9,29,45,2020,1,16,...,normal,normal,normal,normal,normal,normal,none,0,HTTP_FLOOD_v2,flood


In [9]:
df_muestra_unificada[["dataset_name", "attack_family", "attack_t", "attack_a"]].value_counts()

dataset_name     attack_family  attack_t      attack_a
REAPER_WORM_v2   worm           none          0           9994
BLASTER_WORM_v2  worm           none          0           9989
HTTP_FLOOD_v2    flood          none          0           9911
ICMP_FLOOD_v2    flood          none          0           9854
UDP_FLOOD_v2     flood          none          0           8428
RED_WORM_v2      worm           none          0           7801
                                tcp_red_w     1           2199
UDP_FLOOD_v2     flood          udp_f         1           1572
ICMP_FLOOD_v2    flood          icmp_smf      1            121
HTTP_FLOOD_v2    flood          http_f        1             89
ICMP_FLOOD_v2    flood          icmp_f        1             25
BLASTER_WORM_v2  worm           tcp_w32_w     1             11
REAPER_WORM_v2   worm           udp_reaper_w  1              6
Name: count, dtype: int64

In [10]:
df_muestra_unificada.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 87 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   feature_1      60000 non-null  int64  
 1   feature_2      60000 non-null  int64  
 2   feature_3      60000 non-null  int64  
 3   feature_4      60000 non-null  int64  
 4   feature_5      60000 non-null  int64  
 5   feature_6      60000 non-null  int64  
 6   feature_7      60000 non-null  int64  
 7   feature_8      60000 non-null  int64  
 8   feature_9      60000 non-null  int64  
 9   feature_10     60000 non-null  int64  
 10  feature_11     60000 non-null  int64  
 11  feature_12     60000 non-null  int64  
 12  feature_13     60000 non-null  int64  
 13  feature_14     60000 non-null  float64
 14  feature_15     60000 non-null  object 
 15  feature_16     60000 non-null  object 
 16  feature_17     60000 non-null  int64  
 17  feature_18     60000 non-null  int64  
 18  featur

In [12]:
print("Valores únicos de attack_a:")
print(df_muestra_unificada["attack_a"].unique())
print("Valores únicos de attack_t:")
print(df_muestra_unificada["attack_t"].unique())

Valores únicos de attack_a:
[0 1]
Valores únicos de attack_t:
['none' 'tcp_w32_w' 'udp_reaper_w' 'tcp_red_w' 'http_f' 'icmp_smf'
 'icmp_f' 'udp_f']


conda install -c conda-forge pyarrow fastparquet

In [19]:
sample_output_path = PROCESSED_PATH / "dataset_attacks_prueba.parquet"
df_muestra_unificada.to_parquet(sample_output_path, index=False, engine='fastparquet')
print(f"Archivo guardado en: {sample_output_path}")

Archivo guardado en: ..\data\processed\dataset_attacks_prueba.parquet


In [21]:
df_test = pd.read_parquet(sample_output_path, engine='fastparquet')
df_test.shape

(60000, 87)

### Construcción del dataset unificado completo

Una vez verificada la estructura de los subconjuntos y la compatibilidad con el formato Parquet, se procede a realizar la unificación completa de los seis datasets seleccionados. 

In [25]:
output_path = PROCESSED_PATH / "dataset_unificado_limpio.parquet"

In [26]:
CHUNKSIZE = 200000

In [27]:
def optimize_dtypes(df):
    for col in df.select_dtypes(include=["int64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")
    return df

In [28]:
all_chunks = []
for file_name, attack_family in files.items():
    
    print("="*70)
    print(f"Procesando: {file_name}")
    path = RAW_PATH / file_name
    row_counter = 0
    
    for chunk_id, chunk in enumerate(
        pd.read_csv(
            path,
            header=None,
            names=column_names,
            chunksize=CHUNKSIZE
        )
    ):
        
        print(f"Chunk {chunk_id} -> shape: {chunk.shape}")
        # columnas auxiliares
        chunk["dataset_name"] = file_name.replace(".csv", "")
        chunk["attack_family"] = attack_family
        
        # posición original dentro del archivo
        chunk["row_id_source"] = range(
            row_counter,
            row_counter + len(chunk)
        )
        row_counter += len(chunk)
        # optimización de memoria
        chunk = optimize_dtypes(chunk)
        
        all_chunks.append(chunk)

Procesando: BLASTER_WORM_v2.csv
Chunk 0 -> shape: (200000, 85)
Chunk 1 -> shape: (200000, 85)
Chunk 2 -> shape: (200000, 85)
Chunk 3 -> shape: (200000, 85)
Chunk 4 -> shape: (200000, 85)
Chunk 5 -> shape: (200000, 85)
Chunk 6 -> shape: (200000, 85)
Chunk 7 -> shape: (200000, 85)
Chunk 8 -> shape: (200000, 85)
Chunk 9 -> shape: (200000, 85)
Chunk 10 -> shape: (200000, 85)
Chunk 11 -> shape: (200000, 85)
Chunk 12 -> shape: (200000, 85)
Chunk 13 -> shape: (200000, 85)
Chunk 14 -> shape: (200000, 85)
Chunk 15 -> shape: (119478, 85)
Procesando: REAPER_WORM_v2.csv
Chunk 0 -> shape: (200000, 85)
Chunk 1 -> shape: (200000, 85)
Chunk 2 -> shape: (200000, 85)
Chunk 3 -> shape: (200000, 85)
Chunk 4 -> shape: (200000, 85)
Chunk 5 -> shape: (200000, 85)
Chunk 6 -> shape: (200000, 85)
Chunk 7 -> shape: (200000, 85)
Chunk 8 -> shape: (200000, 85)
Chunk 9 -> shape: (200000, 85)
Chunk 10 -> shape: (200000, 85)
Chunk 11 -> shape: (200000, 85)
Chunk 12 -> shape: (200000, 85)
Chunk 13 -> shape: (200000, 8

- BLASTER_WORM_v2: 3,119,478 filas
- REAPER_WORM_v2: 4,673,030 filas
- RED_WORM_v2: 5,637,310 filas
- HTTP_FLOOD_v2: 4,109,117 filas
- ICMP_FLOOD_v2: 4,407,202 filas
- UDP_FLOOD_v2: 630,124 filas

22,576,261 filas

In [29]:
resumen_completo = []
for chunk in all_chunks:
    resumen_completo.append(
        chunk.groupby(
            ["dataset_name", "attack_family", "attack_t", "attack_a"],
            observed=True
        ).size().reset_index(name="n")
    )
resumen_completo_df = (
    pd.concat(resumen_completo, ignore_index=True)
    .groupby(["dataset_name", "attack_family", "attack_t", "attack_a"], observed=True)["n"]
    .sum()
    .reset_index()
    .sort_values(["dataset_name", "attack_a", "attack_t"])
)

display(resumen_completo_df)

,dataset_name,attack_family,attack_t,attack_a,n
0,BLASTER_WORM_v2,worm,none,0,3095187
1,BLASTER_WORM_v2,worm,tcp_w32_w,1,24291
3,HTTP_FLOOD_v2,flood,none,0,4086158
2,HTTP_FLOOD_v2,flood,http_f,1,22959
6,ICMP_FLOOD_v2,flood,none,0,4336095
4,ICMP_FLOOD_v2,flood,icmp_f,1,11628
5,ICMP_FLOOD_v2,flood,icmp_smf,1,59479
7,REAPER_WORM_v2,worm,none,0,4671854
8,REAPER_WORM_v2,worm,udp_reaper_w,1,1176
9,RED_WORM_v2,worm,none,0,4381608


In [30]:
resumen_por_dataset = (
    resumen_completo_df
    .groupby(["dataset_name", "attack_family"], observed=True)
    .agg(
        total_filas=("n", "sum"),
        total_ataques=("n", lambda x: x[resumen_completo_df.loc[x.index, "attack_a"] == 1].sum()),
        total_normales=("n", lambda x: x[resumen_completo_df.loc[x.index, "attack_a"] == 0].sum())
    )
    .reset_index()
)
resumen_por_dataset["attack_ratio"] = (
    resumen_por_dataset["total_ataques"] / resumen_por_dataset["total_filas"]
)
display(resumen_por_dataset)

,dataset_name,attack_family,total_filas,total_ataques,total_normales,attack_ratio
0,BLASTER_WORM_v2,worm,3119478,24291,3095187,0.007787
1,HTTP_FLOOD_v2,flood,4109117,22959,4086158,0.005587
2,ICMP_FLOOD_v2,flood,4407202,71107,4336095,0.016134
3,REAPER_WORM_v2,worm,4673030,1176,4671854,0.000252
4,RED_WORM_v2,worm,5637310,1255702,4381608,0.222748
5,UDP_FLOOD_v2,flood,630124,93583,536541,0.148515


In [31]:
total_filas = sum(len(chunk) for chunk in all_chunks)
total_filas

22576261

In [32]:
df_final = pd.concat(all_chunks, ignore_index=True)
df_final.shape

(22576261, 88)

In [33]:
df_final.to_parquet(
    output_path,
    index=False,
    engine="fastparquet"
)
print(f"Dataset unificado guardado en: {output_path}")

Dataset unificado guardado en: ..\data\processed\dataset_unificado_limpio.parquet


In [ ]:
df_check = pd.read_parquet(output_path, engine="fastparquet")
df_check.shape